# 🔥 Crias Forge — Gerador 3D no Colab (GPU grátis)

Gera modelos `.glb` a partir das imagens (fundo verde já removido ou não).

**Como usar:**
1. Menu `Ambiente de execução` → `Alterar tipo` → **GPU (T4)** → Salvar
2. `Ambiente de execução` → `Executar tudo`
3. Quando a célula 3 pedir, envie as imagens (pode várias de uma vez)
4. No final baixa um `.zip` com todos os `.glb`

Cada imagem leva ~30-60s na T4.

In [ ]:
# 1) Instalação (roda 1x, ~3 min)
!git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q onnxruntime
print('✅ instalação concluída')

In [ ]:
# 2) Remoção de fundo verde chroma (mesma lógica do jogo)
from PIL import Image
import os

def chroma_clean(path):
    im = Image.open(path).convert('RGBA')
    px = im.load()
    w, h = im.size
    for y in range(h):
        for x in range(w):
            r, g, b, a = px[x, y]
            if g > 90 and g > r * 1.35 and g > b * 1.35:
                px[x, y] = (0, 0, 0, 0)
            elif g > max(r, b):
                px[x, y] = (r, max(r, b), b, a)
    im.save(path)
    return path
print('✅ pronto')

In [ ]:
# 3) Envie as imagens (verde chroma ou já transparentes)
from google.colab import files
os.makedirs('inputs', exist_ok=True)
up = files.upload()
paths = []
for name, data in up.items():
    p = os.path.join('inputs', name)
    open(p, 'wb').write(data)
    chroma_clean(p)
    paths.append(p)
print('✅', len(paths), 'imagem(ns) prontas:', paths)

In [ ]:
# 4) Gera os modelos .glb
for p in paths:
    nome = os.path.splitext(os.path.basename(p))[0].lower().replace(' ', '-')
    print('▶ gerando', nome, '…')
    !python run.py "{p}" --output-dir "out/{nome}" --model-save-format glb --bake-texture --texture-resolution 1024
print('✅ geração concluída')

In [ ]:
# 5) Empacota e baixa
import glob, shutil, zipfile
os.makedirs('final', exist_ok=True)
achou = []
for g in glob.glob('out/**/*.glb', recursive=True):
    nome = g.split(os.sep)[1] + '.glb'
    shutil.copy(g, os.path.join('final', nome))
    achou.append(nome)
with zipfile.ZipFile('crias-modelos.zip', 'w') as z:
    for f in achou:
        z.write(os.path.join('final', f), f)
print('✅ modelos:', achou)
from google.colab import files as gf
gf.download('crias-modelos.zip')